# 01 — Dataset Inventory

**Project:** Retail Intelligence — Customer, Sales & Demand Analytics
**Phase:** 1 — Data Understanding
**Dataset:** Olist Brazilian E-Commerce (9 raw CSV files in `data/raw/`)

## Purpose

This notebook only looks at the structure of the nine raw files. It records what is in the data as delivered, so that later phases (cleaning, integration, analysis) start from a documented baseline.

For each file it records:

| # | Attribute | Why it matters |
|---|-----------|----------------|
| 1 | File name | Traceability back to the source |
| 2 | Row count | Volume and granularity of each table |
| 3 | Column count | Width of each table |
| 4 | Column names | Basis for understanding content and join keys |
| 5 | Data types | How pandas read each column (types are guessed, not declared) |
| 6–7 | Missing values (count and %) | Completeness of each column |
| 8 | Duplicate rows | Whether some records appear more than once |
| 9 | Memory usage | How much memory each table needs for later work |

## Ground rules

- Files are **read only** from `data/raw/`. Nothing is written back.
- **No cleaning**: no rows or columns are dropped, no missing values are filled, no types are converted, and no features are created.
- Missing values are **counted, not interpreted**. Why a value is missing is a question for a later phase.
- Each file is loaded **once** and kept in memory for every check.

## 1. Setup

Import the libraries and set display options so wide tables show in full. Only pandas and the standard library are needed.

In [1]:
import hashlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

print(f"Python      : {sys.version.split()[0]}")
print(f"Interpreter : {sys.executable}")
print(f"pandas      : {pd.__version__}")

Python      : 3.12.10
Interpreter : C:\Users\Admin\Desktop\retail-intelligence\.venv\Scripts\python.exe
pandas      : 3.1.0


### 1.1 Confirm the project virtual environment

The notebook should run on the project's `.venv` interpreter, so that results can be reproduced with the versions in `requirements.txt`. This cell only warns if another interpreter is in use; it does not stop the notebook.

In [2]:
IN_PROJECT_VENV = ".venv" in Path(sys.prefix).parts
print("Running inside project .venv:", IN_PROJECT_VENV)
if not IN_PROJECT_VENV:
    print("WARNING: kernel is not the project's .venv — select the 'Retail Intelligence (.venv)' kernel.")

Running inside project .venv: True


## 2. Project paths

Paths are built with `pathlib` and are **relative to the project root**. The root is found by walking up from the current working directory until a folder containing `data/raw/` appears. This way the notebook works whether Jupyter was started from the project root or from `notebooks/`, and no machine-specific absolute path is hard-coded.

In [3]:
def find_project_root(start: Path) -> Path:
    # Return the first directory (start or a parent) that contains data/raw/.
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not locate a project root containing data/raw/ above {start}")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root :", PROJECT_ROOT)
print("Raw data dir :", RAW_DIR.relative_to(PROJECT_ROOT))

Project root : C:\Users\Admin\Desktop\retail-intelligence
Raw data dir : data\raw


## 3. Discover the raw files

The list of expected files is written out explicitly and compared with what is actually in `data/raw/`. Any missing or unexpected file is reported straight away instead of being silently skipped.

In [4]:
EXPECTED_FILES = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]

found_files = sorted(p.name for p in RAW_DIR.glob("*.csv"))
missing_files = sorted(set(EXPECTED_FILES) - set(found_files))
unexpected_files = sorted(set(found_files) - set(EXPECTED_FILES))

print(f"Expected: {len(EXPECTED_FILES)} | Found: {len(found_files)}")
print("Missing   :", missing_files or "none")
print("Unexpected:", unexpected_files or "none")

Expected: 9 | Found: 9
Missing   : none
Unexpected: none


## 4. Fingerprint the original files (before loading)

To show at the end that the raw files were **not changed**, each file's size, last-modified time and SHA-256 hash are recorded **before** any data is read. Section 11 takes the same fingerprint again and compares the two.

In [5]:
def fingerprint(path: Path) -> dict:
    # Size, modification time and SHA-256 of a file, read in 1 MB chunks.
    sha = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            sha.update(chunk)
    stat = path.stat()
    return {"file": path.name, "size_bytes": stat.st_size,
            "modified_ns": stat.st_mtime_ns, "sha256": sha.hexdigest()}

fingerprints_before = pd.DataFrame(
    [fingerprint(RAW_DIR / name) for name in EXPECTED_FILES if (RAW_DIR / name).exists()]
).set_index("file")

display(fingerprints_before[["size_bytes", "sha256"]])

,size_bytes,sha256
file,,
olist_customers_dataset.csv,9033957,983a422239e1712ded753b3bf9ecf47dc73f144d306029...
olist_geolocation_dataset.csv,61273883,b514f6fc991b9566aeba02aa5d67e2c3630f034b60a0e0...
olist_order_items_dataset.csv,15438671,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75f...
olist_order_payments_dataset.csv,5777138,4f713964f2815dbbaa40b9488268c55aac3627bfce5aa9...
olist_order_reviews_dataset.csv,14451670,012b61c7593e34f51fa614efdf802b9c7056ce6aae5307...
olist_orders_dataset.csv,17654914,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c9...
olist_products_dataset.csv,2379446,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b...
olist_sellers_dataset.csv,174703,1f643d2b950373b85735e7794b20986f528d7a000432e7...
product_category_name_translation.csv,2613,a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3...


## 5. Load every dataset (once)

Each CSV is read with `pd.read_csv` using **pandas' default type inference**. No `dtype`, `parse_dates` or `na_values` overrides are passed, so the tables show the data exactly as pandas reads it with no extra interpretation. (Section 9 points out where these defaults are misleading.)

Loading errors are handled per file: the cause is recorded in `load_errors` and the remaining files still load. Every DataFrame is kept in the `datasets` dictionary so later steps reuse it instead of reading the file again.

In [6]:
datasets: dict[str, pd.DataFrame] = {}
load_errors: dict[str, str] = {}

for name in EXPECTED_FILES:
    path = RAW_DIR / name
    try:
        datasets[name] = pd.read_csv(path)
    except FileNotFoundError:
        load_errors[name] = "File not found"
    except pd.errors.EmptyDataError:
        load_errors[name] = "File is empty"
    except pd.errors.ParserError as exc:
        load_errors[name] = f"Parse error: {exc}"
    except UnicodeDecodeError as exc:
        load_errors[name] = f"Encoding error: {exc}"
    except Exception as exc:  # anything unforeseen is still reported, never swallowed silently
        load_errors[name] = f"{type(exc).__name__}: {exc}"

print(f"Loaded successfully: {len(datasets)} / {len(EXPECTED_FILES)}")
if load_errors:
    print("Load errors:")
    for name, err in load_errors.items():
        print(f"  - {name}: {err}")
else:
    print("No loading errors.")

Loaded successfully: 9 / 9
No loading errors.


## 6. Inspection helpers

Two functions hold all the inspection logic, so every dataset is measured the same way.

- `profile_columns(df)` returns one row per column: data type, non-null count, missing count, missing % and number of distinct values. The distinct count is not used for any decisions here. It only helps the reader see what a column is (an identifier, a category, a measurement).
- `profile_dataset(name, df)` returns one summary row per file: shape, total missing cells, **fully duplicated rows** and memory use.

Notes on definitions:

- *Duplicate rows* counts rows that are identical to an earlier row in **every** column (`df.duplicated()`). The first occurrence is not counted.
- *Memory usage* uses `deep=True`, which includes the actual size of string values, not just pointer sizes.

In [7]:
def profile_columns(df: pd.DataFrame) -> pd.DataFrame:
    # One row per column with type, completeness and cardinality.
    n = len(df)
    missing = df.isna().sum()
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": missing,
        "missing_pct": (missing / n * 100).round(2) if n else 0.0,
        "n_unique": df.nunique(dropna=True),
    }).rename_axis("column")


def profile_dataset(name: str, df: pd.DataFrame) -> dict:
    # One summary row describing an entire file.
    total_cells = df.size
    missing_cells = int(df.isna().sum().sum())
    dup_rows = int(df.duplicated().sum())
    return {
        "file": name,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_cells": missing_cells,
        "missing_cells_pct": round(missing_cells / total_cells * 100, 2) if total_cells else 0.0,
        "columns_with_missing": int((df.isna().sum() > 0).sum()),
        "duplicate_rows": dup_rows,
        "duplicate_rows_pct": round(dup_rows / len(df) * 100, 2) if len(df) else 0.0,
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    }

## 7. Per-dataset inspection

For each file, in turn, the notebook shows:

1. Headline figures: rows, columns, duplicate rows and memory.
2. The column profile: names, types, missing count and missing %.
3. The first three rows, to show what the values actually look like.

The profiles are also collected for the combined tables in Section 8.

In [8]:
dataset_summaries: list[dict] = []
column_profiles: dict[str, pd.DataFrame] = {}

for name, df in datasets.items():
    summary = profile_dataset(name, df)
    cols = profile_columns(df)
    dataset_summaries.append(summary)
    column_profiles[name] = cols

    print("=" * 100)
    print(f"FILE: {name}")
    print(f"  Rows: {summary['rows']:,} | Columns: {summary['columns']} | "
          f"Duplicate rows: {summary['duplicate_rows']:,} ({summary['duplicate_rows_pct']}%) | "
          f"Memory: {summary['memory_mb']} MB")
    print("=" * 100)
    display(cols)
    display(df.head(3))

FILE: olist_customers_dataset.csv
  Rows: 99,441 | Columns: 5 | Duplicate rows: 0 (0.0%) | Memory: 11.03 MB


,dtype,non_null,missing,missing_pct,n_unique
column,,,,,
customer_id,str,99441,0,0.00,99441
customer_unique_id,str,99441,0,0.00,96096
customer_zip_code_prefix,int64,99441,0,0.00,14994
customer_city,str,99441,0,0.00,4119
customer_state,str,99441,0,0.00,27


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


FILE: olist_geolocation_dataset.csv
  Rows: 1,000,163 | Columns: 5 | Duplicate rows: 261,831 (26.18%) | Memory: 50.12 MB


,dtype,non_null,missing,missing_pct,n_unique
column,,,,,
geolocation_zip_code_prefix,int64,1000163,0,0.00,19015
geolocation_lat,float64,1000163,0,0.00,717372
geolocation_lng,float64,1000163,0,0.00,717615
geolocation_city,str,1000163,0,0.00,8011
geolocation_state,str,1000163,0,0.00,27


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP


... (17 outputs left out)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


FILE: product_category_name_translation.csv
  Rows: 71 | Columns: 2 | Duplicate rows: 0 (0.0%) | Memory: 0.0 MB


,dtype,non_null,missing,missing_pct,n_unique
column,,,,,
product_category_name,str,71,0,0.00,71
product_category_name_english,str,71,0,0.00,71


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## 8. Consolidated views

### 8.1 Dataset-level summary

One row per file, sorted by row count. This shows the relative size, completeness and duplication of each table at a glance.

In [9]:
summary_df = (pd.DataFrame(dataset_summaries)
              .set_index("file")
              .sort_values("rows", ascending=False))

totals_df = pd.DataFrame([{
    "rows": summary_df["rows"].sum(),
    "columns": summary_df["columns"].sum(),
    "missing_cells": summary_df["missing_cells"].sum(),
    "columns_with_missing": summary_df["columns_with_missing"].sum(),
    "duplicate_rows": summary_df["duplicate_rows"].sum(),
    "memory_mb": round(summary_df["memory_mb"].sum(), 2),
}], index=["TOTAL"])

display(summary_df)
display(totals_df)

,rows,columns,missing_cells,missing_cells_pct,columns_with_missing,duplicate_rows,duplicate_rows_pct,memory_mb
file,,,,,,,,
olist_geolocation_dataset.csv,1000163,5,0,0.00,0,261831,26.18,50.12
olist_order_items_dataset.csv,112650,7,0,0.00,0,0,0.00,18.37
olist_order_payments_dataset.csv,103886,5,0,0.00,0,0,0.00,8.11
olist_customers_dataset.csv,99441,5,0,0.00,0,0,0.00,11.03
olist_orders_dataset.csv,99441,8,4908,0.62,3,0,0.00,21.95
olist_order_reviews_dataset.csv,99224,7,145903,21.01,2,0,0.00,17.84
olist_products_dataset.csv,32951,9,2448,0.83,8,0,0.00,3.73
olist_sellers_dataset.csv,3095,4,0,0.00,0,0,0.00,0.22
product_category_name_translation.csv,71,2,0,0.00,0,0,0.00,0.00


,rows,columns,missing_cells,columns_with_missing,duplicate_rows,memory_mb
TOTAL,1550922,52,153259,13,261831,131.37


### 8.2 All columns across all datasets

A single long table with one row per (file, column) pair. It works as a **data dictionary skeleton**: it can be filtered or sorted to find every column with missing values, every text column, or every column that appears in more than one file.

In [10]:
all_columns_df = (pd.concat(column_profiles, names=["file", "column"])
                  .reset_index())
all_columns_df.insert(2, "position", all_columns_df.groupby("file").cumcount() + 1)

print(f"Total columns across all datasets: {len(all_columns_df)}")
display(all_columns_df)

Total columns across all datasets: 52


,file,column,position,dtype,non_null,missing,missing_pct,n_unique
0,olist_customers_dataset.csv,customer_id,1,str,99441,0,0.00,99441
1,olist_customers_dataset.csv,customer_unique_id,2,str,99441,0,0.00,96096
2,olist_customers_dataset.csv,customer_zip_code_prefix,3,int64,99441,0,0.00,14994
3,olist_customers_dataset.csv,customer_city,4,str,99441,0,0.00,4119
4,olist_customers_dataset.csv,customer_state,5,str,99441,0,0.00,27
5,olist_geolocation_dataset.csv,geolocation_zip_code_prefix,1,int64,1000163,0,0.00,19015
6,olist_geolocation_dataset.csv,geolocation_lat,2,float64,1000163,0,0.00,717372
7,olist_geolocation_dataset.csv,geolocation_lng,3,float64,1000163,0,0.00,717615
8,olist_geolocation_dataset.csv,geolocation_city,4,str,1000163,0,0.00,8011
9,olist_geolocation_dataset.csv,geolocation_state,5,str,1000163,0,0.00,27


### 8.3 Columns with missing values

This is a filtered view of 8.2 showing only the incomplete columns, sorted by missing %. The values are **only counted**. Whether a gap is an error, a structural "not applicable", or an optional field is a question for later phases.

In [11]:
missing_cols_df = (all_columns_df[all_columns_df["missing"] > 0]
                   .sort_values("missing_pct", ascending=False)
                   .reset_index(drop=True))
display(missing_cols_df[["file", "column", "dtype", "missing", "missing_pct"]])

,file,column,dtype,missing,missing_pct
0,olist_order_reviews_dataset.csv,review_comment_title,str,87656,88.34
1,olist_order_reviews_dataset.csv,review_comment_message,str,58247,58.70
2,olist_orders_dataset.csv,order_delivered_customer_date,str,2965,2.98
3,olist_products_dataset.csv,product_name_lenght,float64,610,1.85
4,olist_products_dataset.csv,product_category_name,str,610,1.85
5,olist_products_dataset.csv,product_description_lenght,float64,610,1.85
6,olist_products_dataset.csv,product_photos_qty,float64,610,1.85
7,olist_orders_dataset.csv,order_delivered_carrier_date,str,1783,1.79
8,olist_orders_dataset.csv,order_approved_at,str,160,0.16
9,olist_products_dataset.csv,product_weight_g,float64,2,0.01


### 8.4 Column names shared across files

Columns with the same name in more than one file are **candidate join keys**. This step only records which names are shared. Whether they actually join cleanly (referential integrity) is checked in a later phase.

In [12]:
shared = (all_columns_df.groupby("column")["file"]
          .agg(lambda s: sorted(s))
          .loc[lambda s: s.str.len() > 1])
shared_df = pd.DataFrame({"appears_in_n_files": shared.str.len(), "files": shared.str.join(", ")})
display(shared_df.sort_values("appears_in_n_files", ascending=False))

,appears_in_n_files,files
column,,
order_id,4,"olist_order_items_dataset.csv, olist_order_pay..."
customer_id,2,"olist_customers_dataset.csv, olist_orders_data..."
product_category_name,2,"olist_products_dataset.csv, product_category_n..."
product_id,2,"olist_order_items_dataset.csv, olist_products_..."
seller_id,2,"olist_order_items_dataset.csv, olist_sellers_d..."


## 9. Structural signals for data-quality review

The checks below **only observe** properties of the raw data. They change nothing.

1. **Candidate identifier uniqueness**: for every `*_id` column, are the values unique within the file? This shows the table's grain (one row per X).
2. **Date-like text columns**: text columns whose values look like `YYYY-MM-DD`. With default inference pandas reads them as strings, not datetimes.
3. **Zip-code columns read as numbers**: Brazilian CEP prefixes are 5-digit codes. If read as integers, any leading zero is lost, which shows up as values below 10000.
4. **Column-name spelling**: names are kept exactly as in the source. Misspellings are recorded here so later code refers to them correctly.

In [13]:
# 9.1 Uniqueness of identifier columns within each file
id_rows = []
for name, df in datasets.items():
    for col in [c for c in df.columns if c.endswith("_id")]:
        id_rows.append({"file": name, "column": col, "rows": len(df),
                        "n_unique": df[col].nunique(), "is_unique": df[col].is_unique})
id_uniqueness_df = pd.DataFrame(id_rows)
display(id_uniqueness_df)

,file,column,rows,n_unique,is_unique
0,olist_customers_dataset.csv,customer_id,99441,99441,True
1,olist_customers_dataset.csv,customer_unique_id,99441,96096,False
2,olist_order_items_dataset.csv,order_id,112650,98666,False
3,olist_order_items_dataset.csv,order_item_id,112650,21,False
4,olist_order_items_dataset.csv,product_id,112650,32951,False
5,olist_order_items_dataset.csv,seller_id,112650,3095,False
6,olist_order_payments_dataset.csv,order_id,103886,99440,False
7,olist_order_reviews_dataset.csv,review_id,99224,98410,False
8,olist_order_reviews_dataset.csv,order_id,99224,98673,False
9,olist_orders_dataset.csv,order_id,99441,99441,True


In [14]:
# 9.2 Text columns whose (sampled) non-null values all match a YYYY-MM-DD pattern
date_like = []
for name, df in datasets.items():
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col]):
            sample = df[col].dropna().astype(str).head(1000)
            if len(sample) and sample.str.match(r"^\d{4}-\d{2}-\d{2}").all():
                date_like.append({"file": name, "column": col, "dtype": str(df[col].dtype),
                                  "example": sample.iloc[0]})
date_like_df = pd.DataFrame(date_like)
display(date_like_df)

,file,column,dtype,example
0,olist_order_items_dataset.csv,shipping_limit_date,str,2017-09-19 09:45:35
1,olist_order_reviews_dataset.csv,review_creation_date,str,2018-01-18 00:00:00
2,olist_order_reviews_dataset.csv,review_answer_timestamp,str,2018-01-18 21:46:59
3,olist_orders_dataset.csv,order_purchase_timestamp,str,2017-10-02 10:56:33
4,olist_orders_dataset.csv,order_approved_at,str,2017-10-02 11:07:15
5,olist_orders_dataset.csv,order_delivered_carrier_date,str,2017-10-04 19:55:00
6,olist_orders_dataset.csv,order_delivered_customer_date,str,2017-10-10 21:25:13
7,olist_orders_dataset.csv,order_estimated_delivery_date,str,2017-10-18 00:00:00


In [15]:
# 9.3 Zip-code prefix columns: inferred type and how many values have fewer than 5 digits
zip_rows = []
for name, df in datasets.items():
    for col in [c for c in df.columns if "zip_code_prefix" in c]:
        zip_rows.append({"file": name, "column": col, "dtype": str(df[col].dtype),
                         "min": df[col].min(), "max": df[col].max(),
                         "values_below_10000": int((df[col] < 10000).sum())})
zip_df = pd.DataFrame(zip_rows)
display(zip_df)

,file,column,dtype,min,max,values_below_10000
0,olist_customers_dataset.csv,customer_zip_code_prefix,int64,1003,99990,23995
1,olist_geolocation_dataset.csv,geolocation_zip_code_prefix,int64,1001,99990,245733
2,olist_sellers_dataset.csv,seller_zip_code_prefix,int64,1001,99730,1027


In [16]:
# 9.4 Column names containing the source's misspelling "lenght"
misspelled = all_columns_df[all_columns_df["column"].str.contains("lenght")][["file", "column"]]
display(misspelled)

,file,column
39,olist_products_dataset.csv,product_name_lenght
40,olist_products_dataset.csv,product_description_lenght


In [17]:
# 9.5 Distinct product category names: products table vs. translation table (counts only)
n_cat_products = datasets["olist_products_dataset.csv"]["product_category_name"].nunique()
n_cat_translation = datasets["product_category_name_translation.csv"]["product_category_name"].nunique()
print(f"Distinct categories in products   : {n_cat_products}")
print(f"Distinct categories in translation: {n_cat_translation}")

Distinct categories in products   : 73
Distinct categories in translation: 71


## 10. What each dataset represents

The descriptions below are **inferred from the column names and the profiles above**. They are working hypotheses to be confirmed in later phases.

| File | Appears to represent | Grain (from Section 9.1) | Key columns |
|------|----------------------|--------------------------|-------------|
| `olist_orders_dataset.csv` | **Order header / lifecycle**: one record per order, with its status and the timestamps for purchase → approval → handed to carrier → delivered, plus the estimated delivery date. | One row per `order_id` (unique) | `order_id`, `customer_id` |
| `olist_order_items_dataset.csv` | **Order lines**: the products in each order, which seller supplied each, the item price and freight cost, and the seller's shipping deadline. | One row per (`order_id`, `order_item_id`); `order_id` repeats | `order_id`, `product_id`, `seller_id` |
| `olist_order_payments_dataset.csv` | **Payment transactions**: how each order was paid (type, number of instalments, value). One order can have several payments, numbered by `payment_sequential`. | One row per (`order_id`, `payment_sequential`); `order_id` repeats | `order_id` |
| `olist_order_reviews_dataset.csv` | **Customer reviews**: a 1–5 score, an optional title and comment, the date the review was created and the date it was answered. | `review_id` is **not** unique, so the grain is not simply one row per review | `review_id`, `order_id` |
| `olist_customers_dataset.csv` | **Customers**: customer identifiers and location (zip prefix, city, state). There are two identifiers. `customer_id` is unique per row, while `customer_unique_id` has fewer distinct values, which suggests one person can appear under several `customer_id`s. | One row per `customer_id` | `customer_id`, `customer_unique_id`, `customer_zip_code_prefix` |
| `olist_sellers_dataset.csv` | **Sellers (merchants)**: seller identifier and location. | One row per `seller_id` | `seller_id`, `seller_zip_code_prefix` |
| `olist_products_dataset.csv` | **Product catalogue**: category (in Portuguese), lengths of the listing's name and description, number of photos, and physical weight and dimensions. | One row per `product_id` | `product_id`, `product_category_name` |
| `product_category_name_translation.csv` | **Lookup table**: translates Portuguese category names to English. | One row per category | `product_category_name` |
| `olist_geolocation_dataset.csv` | **Geographic reference**: latitude/longitude points per zip-code prefix, with city and state. Many points share the same prefix. | Many rows per `geolocation_zip_code_prefix` | `geolocation_zip_code_prefix` |

Taken together, the files form a **relational schema** centred on `orders`. Items, payments and reviews link to it by `order_id`, and customers link by `customer_id`. Items link to products and sellers. Zip-code prefixes can link customers and sellers to geolocation, and the translation table adds English names to product categories.

## 11. Verification

### 11.1 All nine datasets were inspected

In [18]:
inspected = set(column_profiles)
checks = {
    "All 9 expected files present on disk": not missing_files,
    "All 9 files loaded without error": len(datasets) == len(EXPECTED_FILES) and not load_errors,
    "All 9 files profiled": inspected == set(EXPECTED_FILES),
    "Summary table has 9 rows": len(summary_df) == len(EXPECTED_FILES),
    "Column table covers every column": len(all_columns_df) == sum(df.shape[1] for df in datasets.values()),
}
for label, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")
assert all(checks.values()), "Inventory incomplete — see FAIL lines above."

[PASS] All 9 expected files present on disk
[PASS] All 9 files loaded without error
[PASS] All 9 files profiled
[PASS] Summary table has 9 rows
[PASS] Column table covers every column


### 11.2 Original files are untouched

The fingerprints are taken again and compared with those from Section 4. A file counts as unchanged only if its size, modification time **and** SHA-256 hash all match.

In [19]:
fingerprints_after = pd.DataFrame(
    [fingerprint(RAW_DIR / name) for name in EXPECTED_FILES]
).set_index("file")

integrity = pd.DataFrame({
    "size_unchanged": fingerprints_before["size_bytes"] == fingerprints_after["size_bytes"],
    "mtime_unchanged": fingerprints_before["modified_ns"] == fingerprints_after["modified_ns"],
    "sha256_unchanged": fingerprints_before["sha256"] == fingerprints_after["sha256"],
})
integrity["untouched"] = integrity.all(axis=1)
display(integrity)

assert integrity["untouched"].all(), "One or more raw files changed during inspection!"
print(f"All {len(integrity)} raw files verified unchanged (size, mtime, SHA-256).")

,size_unchanged,mtime_unchanged,sha256_unchanged,untouched
file,,,,
olist_customers_dataset.csv,True,True,True,True
olist_geolocation_dataset.csv,True,True,True,True
olist_order_items_dataset.csv,True,True,True,True
olist_order_payments_dataset.csv,True,True,True,True
olist_order_reviews_dataset.csv,True,True,True,True
olist_orders_dataset.csv,True,True,True,True
olist_products_dataset.csv,True,True,True,True
olist_sellers_dataset.csv,True,True,True,True
product_category_name_translation.csv,True,True,True,True


All 9 raw files verified unchanged (size, mtime, SHA-256).


## 12. Summary of findings

### Inventory

- **9 / 9** files were found, loaded without error and profiled.
- **1,550,922 rows** and **52 columns** in total, taking about **131 MB** in memory (deep).
- `olist_geolocation_dataset.csv` is by far the largest file (~1.0 M rows, about two-thirds of all rows). The translation table is the smallest (71 rows).

### Initial data-quality observations

These are recorded for review. **None of them have been acted on.**

1. **Duplicate rows only in geolocation.** `olist_geolocation_dataset.csv` has **261,831** rows (~26%) that exactly repeat an earlier row. The other eight files have no exact duplicate rows.
2. **Missing values are in three files only.**
   - `olist_order_reviews_dataset.csv`: `review_comment_title` is missing in ~88% of rows and `review_comment_message` in ~59%.
   - `olist_orders_dataset.csv`: `order_approved_at` (160), `order_delivered_carrier_date` (1,783) and `order_delivered_customer_date` (2,965) have gaps. Whether these relate to `order_status` has **not** been checked.
   - `olist_products_dataset.csv`: the same **610** rows' worth of gaps appear in each of category, name length, description length and photo count. **2** rows' worth appear in weight and each dimension.
   - Customers, sellers, order items, payments, geolocation and the translation table have **no missing values**.
3. **Timestamps are stored as text.** All **8** date/time columns (5 in orders, 1 in order items, 2 in reviews) load as strings with default inference.
4. **Zip-code prefixes lose their leading zeros.** All three zip-prefix columns are inferred as `int64`, and their minimum values are 4-digit numbers (e.g. 1001). A 5-digit prefix such as `01001` therefore loses its leading zero on read. This affects 23,995 customer rows, 245,733 geolocation rows and 1,027 seller rows. These columns will need to be loaded as text in a later phase.
5. **`review_id` is not unique** in the reviews file. The same review identifier appears on more than one row.
6. **`customer_unique_id` has fewer distinct values than `customer_id`**, which points to customers who appear under more than one `customer_id`. This matters for any customer-level analysis.
7. **Category coverage differs.** The products table has 73 distinct category names and the translation table has 71. Some categories may have no English translation (to be confirmed).
8. **Misspelled source column names.** `product_name_lenght` and `product_description_lenght` are spelled this way in the source and have been left unchanged.
9. **Product measurements are floats.** The length/photo-count columns are inferred as `float64` because they contain missing values, even though the values present appear to be whole numbers.

### Next steps (not performed here)

These observations will feed into the data cleaning and integration phase. Per the scope of this task, **no cleaning, type conversion, imputation or feature engineering has been done**.